In [2]:
from dotenv import load_dotenv

load_dotenv()

True

## Summarize messages

In [3]:
from langchain_openrouter import ChatOpenRouter
model = ChatOpenRouter(model="openai/gpt-5-nano")   # or anthropic/claude-3.5-haiku, etc.

In [4]:
from langchain.agents import create_agent
from langgraph.checkpoint.memory import InMemorySaver
from langchain.agents.middleware import SummarizationMiddleware

agent = create_agent(
    # model="gpt-5-nano",
    model=model,
    checkpointer=InMemorySaver(),
    middleware=[
        SummarizationMiddleware(
            # model="gpt-4o-mini",
            model=model,
            trigger=("tokens", 100),
            keep=("messages", 1)
        )
    ],
)

In [5]:
from langchain.messages import HumanMessage, AIMessage
from pprint import pprint

response = agent.invoke(
    {"messages": [
        HumanMessage(content="What is the capital of the moon?"),
        AIMessage(content="The capital of the moon is Lunapolis."),
        HumanMessage(content="What is the weather in Lunapolis?"),
        AIMessage(content="Skies are clear, with a high of 120C and a low of -100C."),
        HumanMessage(content="How many cheese miners live in Lunapolis?"),
        AIMessage(content="There are 100,000 cheese miners living in Lunapolis."),
        HumanMessage(content="Do you think the cheese miners' union will strike?"),
        AIMessage(content="Yes, because they are unhappy with the new president."),
        HumanMessage(content="If you were Lunapolis' new president how would you respond to the cheese miners' union?"),
        ]},
    {"configurable": {"thread_id": "1"}}
)

pprint(response)

{'messages': [HumanMessage(content="Here is a summary of the conversation to date:\n\n## SESSION INTENT\n\n- User aims to obtain fictional information about Lunapolis (capital of the moon), including its weather, population (cheese miners), and labor sentiment (union strike probability).\n\n## SUMMARY\n\n- Lunapolis is stated as the capital of the moon (fictional).\n- Weather: clear skies, high of 120°C and low of -100°C (extreme, fictional).\n- Population: 100,000 cheese miners residing in Lunapolis (fictional).\n- Labor sentiment: yes, the cheese miners' union will strike due to dissatisfaction with the new president (fictional).\n- All information is presented as fictional content.\n\n## ARTIFACTS\n\n- None\n\n## NEXT STEPS\n\n- If desired, provide additional fictional details about Lunapolis (e.g., governance, economy, other demographics) or modify parameters (weather ranges, population size) for new scenarios.", additional_kwargs={'lc_source': 'summarization'}, response_metadata={

In [6]:
print(response["messages"][0].content)

Here is a summary of the conversation to date:

## SESSION INTENT

- User aims to obtain fictional information about Lunapolis (capital of the moon), including its weather, population (cheese miners), and labor sentiment (union strike probability).

## SUMMARY

- Lunapolis is stated as the capital of the moon (fictional).
- Weather: clear skies, high of 120°C and low of -100°C (extreme, fictional).
- Population: 100,000 cheese miners residing in Lunapolis (fictional).
- Labor sentiment: yes, the cheese miners' union will strike due to dissatisfaction with the new president (fictional).
- All information is presented as fictional content.

## ARTIFACTS

- None

## NEXT STEPS

- If desired, provide additional fictional details about Lunapolis (e.g., governance, economy, other demographics) or modify parameters (weather ranges, population size) for new scenarios.


## Trim/delete messages

In [7]:
from typing import Any
from langchain.agents import AgentState
from langchain.messages import RemoveMessage
from langgraph.runtime import Runtime
from langchain.agents.middleware import before_agent
from langchain.messages import ToolMessage

@before_agent
def trim_messages(state: AgentState, runtime: Runtime) -> dict[str, Any] | None:
    """Remove all the tool messages from the state"""
    messages = state["messages"]

    tool_messages = [m for m in messages if isinstance(m, ToolMessage)]
    
    return {"messages": [RemoveMessage(id=m.id) for m in tool_messages]}

In [8]:
agent = create_agent(
    # model="gpt-5-nano",
    model=model,
    checkpointer=InMemorySaver(),
    middleware=[trim_messages],
)

In [9]:
response = agent.invoke(
    {"messages": [
        HumanMessage(content="My device won't turn on. What should I do?"),
        ToolMessage(content="blorp-x7 initiating diagnostic ping…", tool_call_id="1"),
        AIMessage(content="Is the device plugged in and turned on?"),
        HumanMessage(content="Yes, it's plugged in and turned on."),
        ToolMessage(content="temp=42C voltage=2.9v … greeble complete.", tool_call_id="2"),
        AIMessage(content="Is the device showing any lights or indicators?"),
        HumanMessage(content="What's the temperature of the device?")
        ]},
    {"configurable": {"thread_id": "2"}}
)

pprint(response)

{'messages': [HumanMessage(content="My device won't turn on. What should I do?", additional_kwargs={}, response_metadata={}, id='723e415a-4763-4b81-a76c-9715909ed76a'),
              AIMessage(content='Is the device plugged in and turned on?', additional_kwargs={}, response_metadata={}, id='db653b34-7ee9-4035-a87a-17c254efe5e6', tool_calls=[], invalid_tool_calls=[]),
              HumanMessage(content="Yes, it's plugged in and turned on.", additional_kwargs={}, response_metadata={}, id='1977fb27-2cb2-4b11-be1f-4f30f66d273d'),
              AIMessage(content='Is the device showing any lights or indicators?', additional_kwargs={}, response_metadata={}, id='eba7deee-c88e-465e-8d03-01882b6da3e8', tool_calls=[], invalid_tool_calls=[]),
              HumanMessage(content="What's the temperature of the device?", additional_kwargs={}, response_metadata={}, id='e281ab69-f063-48bb-b7fa-04e98a5000eb'),
              AIMessage(content='I can’t read your device’s temperature from here. If you want 

In [10]:
print(response["messages"][-1].content)

I can’t read your device’s temperature from here. If you want to check it yourself, tell me what device you’re using (PC/laptop, smartphone, etc.) and the OS, and I’ll give exact steps. In the meantime here are general options:

- For Windows PCs/Laptops:
  - Use a monitoring tool like HWInfo or CPU-Z/HWMonitor. Install, run, and check CPU/GPU temps (idle vs. load).
  - Or reboot and enter BIOS/UEFI (usually F2, F10, or Delete) to view hardware temps under Hardware Monitor or System Health.

- For macOS:
  - Use a third-party app such as iStat Menus or Macs Fan Control to view CPU temps. macOS doesn’t show temps in a built-in UI.

- For smartphones:
  - Android: use an app like CPU-Z or AIDA64 to read CPU/SoC temps.
  - iPhone: temps aren’t shown in a simple way; you can’t easily read internal temps without specialized tools.

If the device is hot and won’t turn on, let it cool for 15–30 minutes in a well-ventilated area, then try again. Also check:
- The charger and charging cable are